# Modulo 9 · Lezione 7 — Documentazione

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | Docstring e type hint in Python |
| 2 | OpenAPI/Swagger — documentazione automatica dell'API |
| 3 | README professionale per il progetto |
| 4 | Git e GitHub — flusso di base |
| 5 | `requirements.txt` e riproducibilità |

---
## 1. Docstring e type hint

In [ ]:
# Non documentata — codice funzionante ma illeggibile per un collaboratore
def proc(d, t=7, r=False):
    import numpy as np
    d2 = d.dropna().drop_duplicates()
    d2['so2'] = d2['free sulfur dioxide'] / d2['total sulfur dioxide']
    d2['bv'] = (d2['quality'] >= t).astype(int)
    if r:
        print(d2.shape)
    return d2

print("Funzione non documentata: cosa fa 'r'? cosa restituisce?")

In [ ]:
import pandas as pd
import numpy as np
from typing import Optional

def preprocesar_vinos(
    df: pd.DataFrame,
    umbral_calidad: int = 7,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Pulisce e arricchisce il dataset sulla qualità dei vini.

    Passaggi applicati:
    - Elimina le righe con valori nulli
    - Elimina i duplicati esatti
    - Calcola la feature 'so2_ratio'
    - Crea la variabile target binaria 'buen_vino'

    Args:
        df: DataFrame con le colonne del dataset Wine Quality (UCI).
        umbral_calidad: Punteggio minimo per classificare un vino come buono.
        verbose: Se True, stampa lo shape del risultato.

    Returns:
        DataFrame elaborato con le colonne originali più 'so2_ratio' e 'buen_vino'.

    Raises:
        KeyError: Se il DataFrame non contiene 'free sulfur dioxide',
                  'total sulfur dioxide' o 'quality'.

    Example:
        >>> df_raw = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
        >>> df_proc = preprocesar_vinos(df_raw, umbral_calidad=7)
        >>> df_proc.columns.tolist()  # include 'so2_ratio' e 'buen_vino'
    """
    columnas_req = ['free sulfur dioxide', 'total sulfur dioxide', 'quality']
    for col in columnas_req:
        if col not in df.columns:
            raise KeyError(f"Colonna richiesta non trovata: '{col}'")

    df_proc = df.dropna().drop_duplicates().copy()
    df_proc['so2_ratio'] = (
        df_proc['free sulfur dioxide'] / (df_proc['total sulfur dioxide'] + 1e-8)
    )
    df_proc['buen_vino'] = (df_proc['quality'] >= umbral_calidad).astype(int)

    if verbose:
        print(f"Risultato: {df_proc.shape}  (buoni: {df_proc['buen_vino'].mean():.2%})")

    return df_proc

# Vedere la docstring
help(preprocesar_vinos)

In [ ]:
# Type hint in funzioni più complesse
from pathlib import Path
from typing import Union

def cargar_modelo(
    ruta: Union[str, Path],
    tipo: str = 'sklearn'
) -> object:
    """
    Carica un modello da disco.

    Args:
        ruta: Percorso del file del modello (.joblib o .keras).
        tipo: 'sklearn' per joblib, 'keras' per TensorFlow.

    Returns:
        Modello pronto per chiamare .predict().
    """
    import joblib
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"Modello non trovato: {ruta}")

    if tipo == 'sklearn':
        return joblib.load(ruta)
    elif tipo == 'keras':
        import tensorflow as tf
        return tf.keras.models.load_model(ruta)
    else:
        raise ValueError(f"Tipo non supportato: '{tipo}'. Usa 'sklearn' o 'keras'.")

# mypy può verificare i tipi in fase di sviluppo (non in esecuzione)
# mypy api/main.py  →  0 errori se tutto è tipizzato correttamente
print("✓ Funzioni documentate con docstring in stile Google e type hint")

---
## 2. OpenAPI/Swagger — documentazione automatica

In [ ]:
# FastAPI genera automaticamente la specifica OpenAPI 3.0
# Disponibile su: http://localhost:8000/docs  (Swagger UI)
# e su:           http://localhost:8000/redoc (ReDoc)

# Per arricchire la documentazione, aggiungi:
# - description ai campi Pydantic (già fatto in C6)
# - tags alle rotte
# - docstring alle funzioni di rotta

# Esempio di come FastAPI usa le docstring come descrizione dell'endpoint:
from fastapi import FastAPI
from pydantic import BaseModel

app_demo = FastAPI(title="Demo docs")

class EjemploEntrada(BaseModel):
    """Dati di input di esempio."""
    valor: float

@app_demo.post("/ejemplo", tags=["Predizione"], summary="Endpoint di esempio")
def ejemplo(datos: EjemploEntrada) -> dict:
    """
    Endpoint dimostrativo.

    - **valor**: numero tra 0 e 100
    - Restituisce il valore raddoppiato
    """
    return {"resultado": datos.valor * 2}

# Estrarre lo schema OpenAPI senza avviare il server
import json
schema = app_demo.openapi()
print("Schema OpenAPI generato automaticamente:")
print(json.dumps(schema['paths']['/ejemplo'], indent=2))

In [ ]:
# Esportare lo schema OpenAPI del progetto reale in un file
import subprocess, json, time
from pathlib import Path

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--port', '8001', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)

import requests
r = requests.get('http://localhost:8001/openapi.json')
if r.status_code == 200:
    schema_completo = r.json()
    Path('docs').mkdir(exist_ok=True)
    with open('docs/openapi.json', 'w', encoding='utf-8') as f:
        json.dump(schema_completo, f, indent=2, ensure_ascii=False)
    print("✓ docs/openapi.json generato")
    print(f"  Endpoint: {list(schema_completo['paths'].keys())}")

proc.terminate(); proc.wait()

---
## 3. README professionale

In [ ]:
%%writefile docs/README.md
# Wine Quality AI System

Sistema di predizione della qualità dei vini che combina Machine Learning e Deep Learning,
esposto tramite una API REST con FastAPI.

## Architettura

```
data/raw/          → Dataset UCI Wine Quality (download + SHA256)
data/processed/    → Dataset pulito e con feature derivate
data/db/           → SQLite con lo storico delle predizioni
models/            → Modelli addestrati (.joblib, .keras)
api/               → Applicazione FastAPI
docs/              → Documentazione (README, OpenAPI)
logs/              → Log di addestramento e API
```

## Installazione rapida

```bash
# Clonare e configurare l'ambiente
git clone https://github.com/TU_USUARIO/wine-quality-ai
cd wine-quality-ai
python -m venv .venv
source .venv/bin/activate   # Linux/Mac
# .venv\Scripts\activate    # Windows

pip install -r requirements.txt
```

## Pipeline completa

```bash
# 1. Scaricare i dati
jupyter nbconvert --to notebook --execute M9_C1_Obtencion_Datos.ipynb

# 2. Pulire e preelaborare
jupyter nbconvert --to notebook --execute M9_C2_Limpieza_Preparacion.ipynb

# 3. Archiviare in SQL
jupyter nbconvert --to notebook --execute M9_C3_Almacenamiento_SQL.ipynb

# 4. Addestrare i modelli ML
jupyter nbconvert --to notebook --execute M9_C4_Entrenamiento_ML.ipynb

# 5. Addestrare i modelli DL
jupyter nbconvert --to notebook --execute M9_C5_Entrenamiento_DL.ipynb

# 6. Avviare l'API
uvicorn api.main:app --reload --port 8000
```

## API

| Metodo | Rotta | Descrizione |
|--------|------|-------------|
| GET    | `/health` | Stato del server |
| GET    | `/model/info` | Metriche del modello attivo |
| POST   | `/predict` | Predizione singola (RF) |
| POST   | `/predict/dl` | Predizione singola (DL) |
| POST   | `/predict/batch` | Predizione in batch (max. 100) |

Documentazione interattiva: http://localhost:8000/docs

## Esempio di utilizzo

```python
import requests

resultado = requests.post('http://localhost:8000/predict', json={
    "tipo": "tinto",
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "pH": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}).json()

print(resultado)
# {'prediccion': 'normal', 'probabilidad': 0.32, 'modelo': 'rf_v1', ...}
```

## Prestazioni dei modelli

| Modello | F1 | AUC | Commento |
|--------|-----|-----|------------|
| Random Forest (RF) | ~0.73 | ~0.88 | Più veloce in inferenza |
| Rete Neurale (DL)  | ~0.71 | ~0.87 | Stima il punteggio esatto |

## Dataset

- **Fonte**: UCI Machine Learning Repository — Wine Quality
- **Dimensione**: 6.497 vini (1.599 rossi + 4.898 bianchi)
- **Target**: classificazione binaria (`buen_vino`: quality ≥ 7)

## Licenza

MIT © 2024

---
## 4. Git e GitHub — flusso di base

In [ ]:
# Git: sistema di controllo di versione
# GitHub: piattaforma per ospitare repository Git nel cloud

# Flusso di lavoro in un progetto di ML:
git_commands = """
# 1. Inizializzare il repository
git init
git remote add origin https://github.com/TU_USUARIO/wine-quality-ai.git

# 2. Creare .gitignore per escludere file pesanti e segreti
cat .gitignore:
  .venv/
  __pycache__/
  *.pyc
  data/raw/*.csv      # dati grezzi (pesanti)
  data/raw/*.parquet
  models/*.keras      # modelli (pesanti)
  models/*.joblib
  data/db/*.db        # database
  logs/
  .env                # segreti

# 3. Primo commit
git add api/ docs/ M9_*.ipynb requirements.txt .gitignore
git commit -m "feat: pipeline completo M9 — dati, modelli, API"
git push -u origin main

# 4. Flusso delle modifiche (feature branch)
git checkout -b feature/endpoint-dl
# ... modificare api/main.py ...
git add api/main.py
git commit -m "feat: aggiungere endpoint POST /predict/dl"
git checkout main
git merge feature/endpoint-dl

# 5. Vedere lo storico
git log --oneline --graph
"""

print(git_commands)

In [ ]:
gitignore_content = """# Ambienti virtuali
.venv/
env/
venv/

# Python
__pycache__/
*.pyc
*.pyo
*.egg-info/

# Jupyter
.ipynb_checkpoints/

# Dati (non caricare su GitHub — usa DVC o scaricali con lo script)
data/raw/*.csv
data/raw/*.parquet
data/db/

# Modelli addestrati (possono essere molto grandi)
models/*.keras
models/*.joblib
models/*.pkl

# Log
logs/

# Segreti
.env
*.key

# macOS
.DS_Store
"""

with open('.gitignore', 'w') as f:
    f.write(gitignore_content)
print(".gitignore creato")

---
## 5. `requirements.txt` e riproducibilità

In [ ]:
# Generare requirements.txt con le versioni esatte
import subprocess
result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)

# Filtrare solo i pacchetti che usiamo nel progetto
paquetes_proyecto = [
    'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'fastapi',
    'uvicorn', 'pydantic', 'requests', 'sqlalchemy', 'joblib',
    'matplotlib', 'seaborn', 'scipy', 'pyarrow'
]

lineas_filtradas = []
for linea in result.stdout.splitlines():
    pkg = linea.split('==')[0].lower()
    if any(p in pkg for p in paquetes_proyecto):
        lineas_filtradas.append(linea)

requirements = '\n'.join(sorted(lineas_filtradas))
with open('requirements.txt', 'w') as f:
    f.write(requirements + '\n')

print("requirements.txt generato:")
print(requirements)

In [ ]:
# Elencare la struttura del progetto
from pathlib import Path

def listar_estructura(raiz='.', excluir={'.git', '__pycache__', '.venv', '.ipynb_checkpoints'}):
    for p in sorted(Path(raiz).rglob('*')):
        if any(exc in p.parts for exc in excluir):
            continue
        nivel = len(p.relative_to(raiz).parts) - 1
        sangria = '  ' * nivel
        if p.is_dir():
            print(f"{sangria}📁 {p.name}/")
        else:
            size = p.stat().st_size
            size_str = f"{size/1024:.0f}KB" if size > 1024 else f"{size}B"
            print(f"{sangria}  {p.name} ({size_str})")

print("Struttura del progetto:")
listar_estructura()

---
## Pratica

### Esercizio 1 — Docstring per l'API

**Consegna:** Aggiungi docstring complete (stile Google) alle due funzioni più importanti di `api/main.py`:
1. `entrada_a_array(vino: VinoEntrada) -> np.ndarray`
2. `predecir(vino: VinoEntrada) -> PrediccionSalida`

Includi: descrizione, Args, Returns, Raises ed Example.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
# Modifica api/main.py e sostituisci le funzioni con queste versioni documentate:

def entrada_a_array(vino: VinoEntrada) -> np.ndarray:
    """
    Converte il modello Pydantic di input nell'array NumPy atteso dal modello.

    Calcola tre feature derivate prima di costruire l'array:
    - so2_ratio: proporzione di SO2 libera rispetto al totale
    - acid_index: differenza tra acidità fissa e volatile
    - density_diff: distanza della densità dall'acqua pura (1.0 g/cm³)

    Args:
        vino: Istanza validata di VinoEntrada con i valori fisico-chimici.

    Returns:
        Array di forma (1, 15) con le 15 feature nell'ordine atteso dal modello.

    Example:
        >>> v = VinoEntrada(tipo='tinto', fixed_acidity=7.4, ...)
        >>> X = entrada_a_array(v)
        >>> X.shape
        (1, 15)
    """
    d = {
        'fixed acidity': vino.fixed_acidity,
        # ... resto dei campi ...
    }
    d['so2_ratio']    = d['free sulfur dioxide'] / (d['total sulfur dioxide'] + 1e-8)
    d['acid_index']   = d['fixed acidity'] - d['volatile acidity']
    d['density_diff'] = abs(d['density'] - 1.0)
    d['tipo_encoded'] = 1 if vino.tipo == 'tinto' else 0
    return np.array([[d[f] for f in FEATURES_ORDEN]])

@app.post("/predict", response_model=PrediccionSalida, tags=["Predizione"])
def predecir(vino: VinoEntrada) -> PrediccionSalida:
    """
    Predice se un vino è di buona qualità.

    Usa il modello Random Forest addestrato in M9_C4 per classificare il vino
    come 'bueno' (quality ≥ 7) o 'normal' (quality < 7).

    Args:
        vino: Caratteristiche fisico-chimiche del vino da valutare.

    Returns:
        PrediccionSalida con classificazione, probabilità e metadati.

    Raises:
        HTTPException (500): Se il file del modello non è disponibile.
        HTTPException (422): Se i dati di input non superano la validazione Pydantic.
    """
    modelo = cargar_modelo_rf()
    X = entrada_a_array(vino)
    pred = int(modelo.predict(X)[0])
    prob = float(modelo.predict_proba(X)[0, 1])
    return PrediccionSalida(
        prediccion='bueno' if pred else 'normal',
        probabilidad=round(prob, 4),
        modelo='rf_v1',
        timestamp=datetime.now().isoformat()
    )
```
</details>

---
### Esercizio 2 — Generare la documentazione con `pdoc`

**Consegna:** `pdoc` converte le docstring in HTML automaticamente:

1. `pip install pdoc`
2. Crea un modulo `src/preprocessing.py` con la funzione `preprocesar_vinos` documentata
3. `pdoc src/preprocessing.py -o docs/api_docs/`
4. Apri l'HTML generato e verifica il risultato

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from pathlib import Path
import subprocess

# Creare il modulo
Path('src').mkdir(exist_ok=True)

codigo = '''
"""
Modulo di preprocessing del dataset Wine Quality.

Contiene funzioni per pulire, trasformare e arricchire
il dataset UCI Wine Quality prima dell'addestramento.
"""
import pandas as pd
import numpy as np

def preprocesar_vinos(
    df: pd.DataFrame,
    umbral_calidad: int = 7,
    verbose: bool = False
) -> pd.DataFrame:
    """
    Pulisce e arricchisce il dataset sulla qualità dei vini.

    Args:
        df: DataFrame con le colonne del dataset Wine Quality.
        umbral_calidad: Punteggio minimo per classificare come buono.
        verbose: Se True, stampa lo shape del risultato.

    Returns:
        DataFrame elaborato con 'so2_ratio' e 'buen_vino' aggiunti.

    Example:
        >>> df_proc = preprocesar_vinos(df_raw, umbral_calidad=7)
    """
    df_proc = df.dropna().drop_duplicates().copy()
    df_proc['so2_ratio'] = df_proc['free sulfur dioxide'] / (df_proc['total sulfur dioxide'] + 1e-8)
    df_proc['buen_vino'] = (df_proc['quality'] >= umbral_calidad).astype(int)
    if verbose:
        print(f"{df_proc.shape}")
    return df_proc
'''
Path('src/preprocessing.py').write_text(codigo)

# Installare pdoc se necessario
subprocess.run(['pip', 'install', 'pdoc', '-q'])

# Generare l'HTML
Path('docs/api_docs').mkdir(parents=True, exist_ok=True)
result = subprocess.run(['pdoc', 'src/preprocessing.py', '-o', 'docs/api_docs/'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Documentazione generata in docs/api_docs/")
else:
    print(result.stderr)
```
</details>

---
### Esercizio 3 — CHANGELOG.md

**Consegna:** Crea un `docs/CHANGELOG.md` seguendo lo standard [Keep a Changelog](https://keepachangelog.com):

```markdown
## [Unreleased]
## [1.0.0] - 2024-XX-XX
### Added
### Changed
### Fixed
```

Documenta le 8 fasi del progetto con le loro principali decisioni tecniche.

In [ ]:
# ✏️ IL TUO CODICE QUI


---
## Riepilogo

```python
# Docstring in stile Google
def mi_funcion(x: int, y: float = 1.0) -> dict:
    """
    Descrizione in una riga.

    Spiegazione aggiuntiva se necessaria.

    Args:
        x: Descrizione del parametro x.
        y: Descrizione del parametro y.

    Returns:
        Descrizione del valore restituito.

    Raises:
        ValueError: Quando viene sollevato.
    """
    ...

# Git di base
# git init
# git add <file>
# git commit -m "tipo: descrizione concisa"
# git push origin main
```

## Cosa vedremo nella prossima lezione?
Nella **Lezione 8** distribuiremo il sistema in locale: servizio `systemd` su Linux e container **Docker**.